# Nested Grouped Validation

## Imports

In [ ]:
import os
import sys
import time
import pickle
import itertools
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append('../src')
from zoo import TASKS, FAMILY, META, SEED, reg_models, clf_models, make_pipeline, grid_prefix, load_task, make_folds

warnings.filterwarnings('ignore')
sns.set_theme(style='ticks', context='notebook')

from sklearn.base import clone
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, f1_score

## Settings

In [ ]:
# whole studies are held out: 5 outer folds, repeated with different assignments of studies to folds
TASKS_TO_RUN = list(TASKS)
REPEATS = {'MaxLoad': 3, 'LSS': 2, 'FailureMode': 2}
N_OUTER = 5
ONLY_MODELS = None            # for example ['Ridge', 'RF'] for a quick test
OUT = '../results/nested'
os.makedirs(OUT, exist_ok=True)

## Fold Assignment

In [ ]:
df, X, y, groups, is_clf, target = load_task('CFRP_CFRP__LSS')
folds = make_folds(groups, N_OUTER, 1000)
pd.DataFrame({'records': np.bincount(folds), 'studies': [len(set(groups[folds == k])) for k in range(N_OUTER)]})

## Nested Loop

In [ ]:
def run_task(task):
    df, X, y, groups, is_clf, target = load_task(task)
    scoring = 'f1_macro' if is_clf else 'r2'
    n_repeats = REPEATS[task.split('__')[1]]
    models = clf_models() if is_clf else reg_models()
    if ONLY_MODELS:
        models = {k: v for k, v in models.items() if k in ONLY_MODELS}

    res = dict(name=task, clf=is_clf, y=y, groups=groups, feat=list(X.columns), rowid=df['Row_ID'].values,
               vt=df['Value_Type'].values, derived=df['Target_Is_Derived'].values, pred={}, imp={}, fold={})
    for rep in range(n_repeats):
        fold_id = make_folds(groups, N_OUTER, 1000 + rep)
        res['fold'][rep] = fold_id
        for name, (model, param_grid, impute, scale_target) in models.items():
            t0 = time.time()
            prefix = grid_prefix(scale_target, is_clf)
            pred = np.full(len(df), np.nan)
            importance = []
            for k in range(N_OUTER):
                train = np.where(fold_id != k)[0]
                test = np.where(fold_id == k)[0]
                # tuning on the four training folds, again grouped by study
                grid = GridSearchCV(make_pipeline(model, impute, scale_target, is_clf), {prefix + a: v for a, v in param_grid.items()},
                                    cv=GroupKFold(min(3, len(set(groups[train])))), scoring=scoring, n_jobs=2, refit=True, error_score=np.nan)
                grid.fit(X.iloc[train], y[train], groups=groups[train])
                pred[test] = np.ravel(grid.predict(X.iloc[test]))
                try:
                    pi = permutation_importance(grid.best_estimator_, X.iloc[test], y[test], scoring=scoring, n_repeats=3, random_state=0, n_jobs=1)
                    importance.append(pi.importances_mean)
                except Exception:
                    importance.append(np.full(X.shape[1], np.nan))
            res['pred'].setdefault(name, {})[rep] = pred
            res['imp'].setdefault(name, {})[rep] = np.array(importance)
            score = f1_score(y, pred.astype(int), average='macro') if is_clf else r2_score(y, pred)
            print(task, rep, name, f'nested pooled {score:.3f}', f'{time.time() - t0:.0f}s')
        with open(f'{OUT}/{task}.pkl', 'wb') as fh:
            pickle.dump(res, fh)
    return res

In [ ]:
for task in TASKS_TO_RUN:
    run_task(task)